# Validación con la temporada en curso, y una predicción real de la próxima semana

## Qué responde este notebook

Todo lo anterior en Fase 5 valida contra temporadas ya cerradas (2016-2025) — la prueba de que el
modelo generaliza a datos que nunca vio, pero sigue siendo un backtest histórico. Este notebook
lleva el modelo hasta el límite real: **la temporada 2026, en curso ahora mismo**. Dos pasos, en
el orden correcto — entrenado → testeado → validado con lo más reciente:

1. Validar contra las semanas de 2026 que ya se jugaron (dato genuinamente nuevo, ni existía
   cuando se construyó el feature set de Fase 4).
2. Usar el estado real de hoy (última semana jugada + alineación actual de cada equipo) para
   generar una predicción real de la semana que sigue, todavía sin jugarse.

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling

import numpy as np
import pandas as pd
from xgboost import XGBRegressor

pd.set_option("display.width", 160)

MODELOS = {
    "receptions": XGBRegressor(max_depth=4, n_estimators=200, learning_rate=0.03, random_state=0, n_jobs=-1),
    "receiving_yards": XGBRegressor(max_depth=3, n_estimators=200, learning_rate=0.03, random_state=0, n_jobs=-1),
    "receiving_tds": XGBRegressor(objective="reg:tweedie", tweedie_variance_power=1.5, random_state=0, n_jobs=-1),
}

## 1. ¿Dónde va la temporada 2026 ahora mismo?

In [2]:
stats_2026 = data.cargar_stats_semanales([2026])
wr_2026 = stats_2026[stats_2026["position"] == "WR"]
ultima_semana_jugada = int(wr_2026["week"].max())
proxima_semana = ultima_semana_jugada + 1
print(f"última semana jugada en 2026: {ultima_semana_jugada} · próxima semana a predecir: {proxima_semana}")
print(f"filas WR ya jugadas en 2026: {wr_2026.shape[0]}")

última semana jugada en 2026: 3 · próxima semana a predecir: 4
filas WR ya jugadas en 2026: 457


## 2. Validar contra 2026 ya jugado — el dato más fresco posible

Estas semanas no existían cuando se construyó el feature set (Fase 4) ni cuando se compararon los
modelos (`13`/`14`) — es la validación más exigente de todas, más que el propio test 2024-2025.
Se entrena con toda la historia hasta 2025 (nunca se le muestra 2026 en el entrenamiento) y se
predicen las semanas 1 a la última jugada de 2026.

In [3]:
wr_hist = features.construir_tabla_modelado(range(2016, 2027))
features_arbol = features.columnas_por_modelo("arbol")
X_hist = wr_hist[features_arbol]

train_mask = wr_hist["season"] <= 2025
val_2026_mask = (wr_hist["season"] == 2026) & (~wr_hist["es_prediccion"])
print(f"entrenamiento: {train_mask.sum()} filas (2016-2025) · validación 2026: {val_2026_mask.sum()} filas")

metricas_2026 = {}
for target, modelo in MODELOS.items():
    modelo.fit(X_hist[train_mask], wr_hist[target][train_mask])
    pred = modelo.predict(X_hist[val_2026_mask])
    metricas_2026[target] = modeling.metricas(wr_hist[target][val_2026_mask], pred)
pd.DataFrame(metricas_2026).T.round(4)

entrenamiento: 23610 filas (2016-2025) · validación 2026: 457 filas


,mae,rmse,r2
receptions,1.3597,1.8347,0.4257
receiving_yards,20.3911,29.1632,0.3307
receiving_tds,0.2466,0.4838,-0.0731


Comparado contra el test 2024-2025 de `13_comparacion_modelos.ipynb`/`14_receiving_tds_modelos_de_conteo.ipynb`
(MAE 1.37 recepciones, 20.6 yardas, 0.25 TDs), el desempeño sobre 2026 **se mantiene en el mismo
rango, sin degradarse** — es evidencia adicional de estabilidad, ahora contra datos que no existían
en ningún momento anterior de este proyecto.

## 3. Construir la fila "hoy" de cada receptor activo

Para predecir una semana que no se ha jugado no hay fila real de la que partir -- se arma una fila
*placeholder* por jugador (estadísticas en blanco, `season`/`week`/`team` conocidos) usando su
equipo **actual** (`cargar_depth_charts_actual()`, el esquema que sí cubre 2025 en adelante, a
diferencia del histórico). `features.construir_tabla_modelado()` la procesa con la misma lógica de
siempre (`agregar_promedios_jugador`, etc.) — como esa fila queda cronológicamente al final de la
historia real de cada jugador, sus promedios rezagados salen calculados correctamente con
`shift(1)`, sin escribir una segunda versión de esa lógica solo para este caso.

In [4]:
depth_actual = data.cargar_depth_charts_actual([2026], posicion="WR")
snapshot = depth_actual["dt"].max()
print(f"snapshot de depth chart usado: {snapshot}")

equipo_hoy = depth_actual[depth_actual["dt"] == snapshot][["gsis_id", "team", "player_name"]].drop_duplicates("gsis_id")
jugadores_activos = wr_2026["player_id"].unique()
equipo_hoy = equipo_hoy[equipo_hoy["gsis_id"].isin(jugadores_activos)]
print(f"WR con al menos un partido real en 2026: {len(jugadores_activos)}")
print(f"de esos, con equipo confirmado en el depth chart de hoy: {len(equipo_hoy)}")

columnas_stat = [c for c in features.COLUMNAS_BASE_MODELADO
                 if c not in ("yards_per_target", "catch_rate", "air_yards_per_target", "racr_acotado")]
filas_extra = pd.DataFrame({
    "player_id": equipo_hoy["gsis_id"].values,
    "player_display_name": equipo_hoy["player_name"].values,
    "team": equipo_hoy["team"].values,
    "season": 2026,
    "week": proxima_semana,
    "position": "WR",
})
for col in columnas_stat:
    filas_extra[col] = np.nan
print("filas placeholder construidas:", filas_extra.shape)

snapshot de depth chart usado: 2026-09-29 13:59:18+00:00
WR con al menos un partido real en 2026: 188
de esos, con equipo confirmado en el depth chart de hoy: 177
filas placeholder construidas: (177, 23)


## 4. Correr el pipeline completo con las filas placeholder incluidas

In [5]:
wr_con_prediccion = features.construir_tabla_modelado(range(2016, 2027), filas_extra=filas_extra)
filas_pred = wr_con_prediccion[wr_con_prediccion["es_prediccion"]]
print("filas de predicción, tras el pipeline completo:", filas_pred.shape[0])
print("\n% de nulos en las features de árbol, para estas filas:")
print(filas_pred[features_arbol].isna().mean().sort_values(ascending=False).head(5))

filas de predicción, tras el pipeline completo: 177

% de nulos en las features de árbol, para estas filas:
depth_team                    1.000000
receiving_epa_last3_avg       0.079096
catch_rate_career_avg         0.033898
air_yards_share_career_avg    0.000000
target_share_career_avg       0.000000
dtype: float64


`depth_team` sale 100% nulo para estas filas -- es la misma limitación ya documentada desde Fase 2
(el esquema de depth chart de 2025 en adelante no comparte columna de semana con el histórico, no
se ha unificado). XGBoost lo maneja de forma nativa (aprende una dirección para los valores
faltantes), pero es una limitación real que vale la pena resolver antes de usar esto en
producción real -- ver alcance diferido en el plan de Fase 5.

## 5. La predicción real — próxima semana, temporada 2026

Punto estimado de los 3 targets, para los receptores activos con equipo confirmado hoy. Se ordena
por yardas esperadas.

In [6]:
X_final_train = wr_con_prediccion.loc[~wr_con_prediccion["es_prediccion"], features_arbol]
X_pred = filas_pred[features_arbol]

modelos_finales = {}
resultado = filas_pred[["player_display_name", "team"]].copy()
resultado["semana"] = proxima_semana
for target, modelo in MODELOS.items():
    modelo_final = modelo.__class__(**modelo.get_params())
    modelo_final.fit(X_final_train, wr_con_prediccion.loc[~wr_con_prediccion["es_prediccion"], target])
    modelos_finales[target] = modelo_final
    resultado[f"pred_{target}"] = modelo_final.predict(X_pred).round(2)

resultado = resultado.sort_values("pred_receiving_yards", ascending=False).reset_index(drop=True)
print(f"Predicción generada con datos de nflreadpy frescos al {snapshot} -- semana {proxima_semana} de 2026, "
      f"todavía no jugada.")
resultado.head(15)

Predicción generada con datos de nflreadpy frescos al 2026-09-29 13:59:18+00:00 -- semana 4 de 2026, todavía no jugada.


,player_display_name,team,semana,pred_receptions,pred_receiving_yards,pred_receiving_tds
0,Jaxon Smith-Njigba,SEA,4,8.62,93.919998,0.38
1,Amon-Ra St. Brown,DET,4,6.68,88.040001,1.34
2,Ja'Marr Chase,CIN,4,6.19,81.379997,0.55
3,Chris Olave,NO,4,8.18,81.370003,0.53
4,Puka Nacua,LA,4,6.14,80.250000,0.16
5,CeeDee Lamb,DAL,4,6.22,76.230003,0.15
6,Zay Flowers,BAL,4,5.31,74.910004,0.13
7,Justin Jefferson,MIN,4,5.72,71.470001,0.34
8,Garrett Wilson,NYJ,4,5.00,69.639999,0.23
9,Drake London,ATL,4,4.70,66.250000,0.32


## 6. Guardar los modelos ganadores

Los 3 modelos de arriba —entrenados con **toda** la historia disponible (2016-2025), los mismos
que generaron la predicción real de la sección 5— se guardan en `weekly/wr/models/`, junto con la
lista exacta de features (el orden importa) y las métricas de referencia ya obtenidas en el resto
de la fase. No es todavía el Model Card formal de Fase 6/7 (eso necesita más que el archivo del
modelo), pero sí es el modelo real, listo para recargarse sin tener que reentrenar.

In [7]:
import json
import os

os.makedirs("../models", exist_ok=True)

METRICAS_REFERENCIA = {
    "receptions": {"mae_test_2024_2025": 1.372, "r2_test_2024_2025": 0.445,
                   "mae_legado_2025_real": 1.722, "mae_nuevo_2025_real": 1.644},
    "receiving_yards": {"mae_test_2024_2025": 20.647, "r2_test_2024_2025": 0.363,
                         "mae_legado_2025_real": 27.049, "mae_nuevo_2025_real": 25.113},
    "receiving_tds": {"mae_test_2024_2025": 0.2466, "r2_test_2024_2025": -0.045,
                       "mae_legado_2025_real": 0.412, "mae_nuevo_2025_real": 0.341},
}

for target, modelo_final in modelos_finales.items():
    ruta_modelo = f"../models/{target}_xgboost.json"
    modelo_final.save_model(ruta_modelo)

    metadata = {
        "target": target,
        "modelo": type(modelo_final).__name__,
        "hiperparametros": {k: v for k, v in modelo_final.get_params().items() if v is not None},
        "features_arbol_en_orden": features_arbol,
        "entrenado_con": "2016-2025 (toda la historia disponible al momento del guardado)",
        "fecha_de_guardado": pd.Timestamp.now(tz="UTC").isoformat(),
        "snapshot_nflreadpy_usado_en_prediccion": str(snapshot),
        "metricas_referencia": METRICAS_REFERENCIA[target],
        "limitaciones_conocidas": [
            "depth_team sale 100% nulo para 2025 en adelante (esquema de depth chart sin unificar, ver HALLAZGOS.md)",
            "el modelo subestima sistematicamente las semanas boom (ver 13_comparacion_modelos.ipynb y 17_explicabilidad_shap.ipynb)",
        ],
    }
    with open(f"../models/{target}_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

print("guardados:", sorted(os.listdir("../models")))

guardados: ['receiving_tds_metadata.json', 'receiving_tds_xgboost.json', 'receiving_yards_metadata.json', 'receiving_yards_xgboost.json', 'receptions_metadata.json', 'receptions_xgboost.json']


Verificación de que lo guardado es exactamente lo mismo que se usó para predecir -- se recarga
cada modelo desde el archivo y se comparan sus predicciones contra las ya generadas arriba, fila
por fila.

In [8]:
import xgboost as xgb

dmatriz_pred = xgb.DMatrix(X_pred)
for target, modelo_final in modelos_finales.items():
    recargado = xgb.Booster()
    recargado.load_model(f"../models/{target}_xgboost.json")
    pred_original = modelo_final.predict(X_pred)
    pred_recargado = recargado.predict(dmatriz_pred)
    identicas = np.allclose(pred_original, pred_recargado)
    print(f"{target}: predicciones idénticas tras recargar desde disco -- {identicas}")
    assert identicas, f"el modelo recargado de {target} no coincide con el original" 

receptions: predicciones idénticas tras recargar desde disco -- True
receiving_yards: predicciones idénticas tras recargar desde disco -- True
receiving_tds: predicciones idénticas tras recargar desde disco -- True


## Cierre

El modelo entrenado con historia hasta 2025 se valida bien contra 2026, dato que nunca vio en
ningún momento del proyecto, produce una predicción real para la próxima semana de la temporada
en curso, y ya queda guardado en `weekly/wr/models/` -- no hay que reentrenar para volver a usarlo.
Es una corrida de una sola vez en el sentido de que no hay todavía una orquestación recurrente que
la repita sola cada semana (eso sigue siendo Fase 6/7: `tracking.csv`, orquestación
semanal, Model Cards formales) -- pero cierra el círculo entre "entrenamos con el pasado" y "sirve
para predecir lo que viene", que es la pregunta de fondo detrás de todo este proyecto.

Siguiente: [`19_cierre_fase5.ipynb`](19_cierre_fase5.ipynb).